# Module 2: Coalition Structure & Bipartisanship

**Question:** What coalitions exist and how partisan are they?

Analyses:
1. Community deep-dive with richer profiling
2. Bipartisanship index per document and policy area
3. Cross-party bridge legislators
4. Coalition stability across policy areas

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))

import pandas as pd
import numpy as np
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from collections import defaultdict
from networkx.algorithms.community import greedy_modularity_communities

import analysis_utils as au

docs_df = au.load_comprehensive_df()
cosp_df = au.load_cosponsors_df()
communities = au.load_communities()

# Build core graphs
G_cc = au.build_cosponsor_cosponsor_graph(cosp_df, min_shared=2)
print(f"Cosponsor-cosponsor graph: {G_cc.number_of_nodes()} nodes, {G_cc.number_of_edges()} edges")
print(f"Pre-computed communities: {len(communities)}")

## 2.1 Community Deep-Dive

Profile each pre-computed community by: party composition, chamber mix, state diversity, top policy areas, and size.

In [ ]:
# Map bioguide -> info from cosponsor data
bio_info = cosp_df.drop_duplicates("Cosponsor_BioguideId").set_index("Cosponsor_BioguideId")

# Merge cosponsors with policy areas
cosp_policy = cosp_df.merge(docs_df[["AGORA ID", "Policy_Area"]], on="AGORA ID", how="left")

# Profile each community
community_profiles = []
for comm in communities:
    members = comm.get("members", [])
    if not members:
        continue
    
    member_info = bio_info.loc[bio_info.index.isin(members)]
    
    # Party composition
    parties = member_info["Cosponsor_Party"].value_counts()
    d_pct = parties.get("D", 0) / len(members) * 100 if members else 0
    r_pct = parties.get("R", 0) / len(members) * 100 if members else 0
    
    # Chamber mix
    chambers = member_info["Cosponsor_FullName"].apply(
        lambda x: "Senate" if str(x).startswith("Sen.") else "House"
    ).value_counts()
    
    # State diversity
    n_states = member_info["Cosponsor_State"].nunique()
    
    # Top policy areas for this community's members
    comm_docs = cosp_policy[cosp_policy["Cosponsor_BioguideId"].isin(members)]
    top_policies = comm_docs["Policy_Area"].value_counts().head(3)
    
    community_profiles.append({
        "id": comm.get("id", ""),
        "size": len(members),
        "D%": round(d_pct, 1),
        "R%": round(r_pct, 1),
        "Senate": chambers.get("Senate", 0),
        "House": chambers.get("House", 0),
        "States": n_states,
        "Top Policy 1": f"{top_policies.index[0]} ({top_policies.iloc[0]})" if len(top_policies) > 0 else "",
        "Top Policy 2": f"{top_policies.index[1]} ({top_policies.iloc[1]})" if len(top_policies) > 1 else "",
        "Top Policy 3": f"{top_policies.index[2]} ({top_policies.iloc[2]})" if len(top_policies) > 2 else "",
    })

profiles_df = pd.DataFrame(community_profiles)
profiles_df

## 2.2 Bipartisanship Index

Per document: `min(D_cosponsors, R_cosponsors) / total_cosponsors`. 0 = purely partisan, 0.5 = perfectly balanced. Which policy areas produce bipartisan bills?

In [ ]:
# Compute bipartisanship index per document
doc_bipartisan = []
for aid, group in cosp_df.groupby("AGORA ID"):
    bi = au.bipartisanship_index(group)
    policy = docs_df.loc[docs_df["AGORA ID"] == aid, "Policy_Area"]
    pa = policy.values[0] if not policy.empty and pd.notna(policy.values[0]) else "Unknown"
    doc_bipartisan.append({
        "AGORA ID": aid,
        "bipartisanship": bi,
        "policy_area": pa,
        "n_cosponsors": len(group),
        "n_D": (group["Cosponsor_Party"] == "D").sum(),
        "n_R": (group["Cosponsor_Party"] == "R").sum(),
    })

bi_df = pd.DataFrame(doc_bipartisan)

# Overall distribution
fig_bi_hist = px.histogram(
    bi_df, x="bipartisanship", nbins=30,
    labels={"bipartisanship": "Bipartisanship Index"},
    title="Distribution of Bipartisanship Across AI Policy Bills",
    color_discrete_sequence=["#636EFA"],
)
fig_bi_hist.add_vline(x=bi_df["bipartisanship"].median(), line_dash="dash",
                       annotation_text=f"Median: {bi_df['bipartisanship'].median():.2f}")
fig_bi_hist.update_layout(template="plotly_white", height=400)
fig_bi_hist.show()

# By policy area (box plot, top 10 areas)
top_areas = bi_df["policy_area"].value_counts().head(10).index
bi_top = bi_df[bi_df["policy_area"].isin(top_areas)]

fig_bi_box = px.box(
    bi_top, x="policy_area", y="bipartisanship",
    color="policy_area",
    labels={"bipartisanship": "Bipartisanship Index", "policy_area": "Policy Area"},
    title="Bipartisanship by Policy Area (Top 10 by Volume)",
)
fig_bi_box.update_layout(template="plotly_white", height=450, showlegend=False,
                          xaxis_tickangle=-30)
fig_bi_box.show()

print(f"Overall median bipartisanship: {bi_df['bipartisanship'].median():.3f}")
print(f"Bills with perfect bipartisan balance (>0.45): {(bi_df['bipartisanship'] > 0.45).sum()}")
print(f"Purely partisan bills (=0): {(bi_df['bipartisanship'] == 0).sum()}")

## 2.3 Cross-Party Bridge Legislators

Identify sponsors with high betweenness centrality in the cross-party subgraph — these are legislators who frequently cosponsor across party lines and serve as connective tissue between partisan blocs.

In [ ]:
# Cross-party subgraph
G_cross = au.cross_party_subgraph(G_cc)
print(f"Cross-party subgraph: {G_cross.number_of_nodes()} nodes, {G_cross.number_of_edges()} edges")
print(f"  (vs full graph: {G_cc.number_of_nodes()} nodes, {G_cc.number_of_edges()} edges)")

# Betweenness centrality in cross-party graph
cross_between = nx.betweenness_centrality(G_cross, weight="weight")
cross_degree = dict(G_cross.degree(weight="weight"))

bridge_df = pd.DataFrame([
    {"bioguide": n,
     "name": G_cross.nodes[n].get("name", "")[:40],
     "party": G_cross.nodes[n].get("party", ""),
     "state": G_cross.nodes[n].get("state", ""),
     "cross_betweenness": cross_between.get(n, 0),
     "cross_weighted_degree": cross_degree.get(n, 0),
     "total_degree": G_cc.degree(n, weight="weight") if G_cc.has_node(n) else 0,
     "cross_ratio": cross_degree.get(n, 0) / max(G_cc.degree(n, weight="weight"), 1) if G_cc.has_node(n) else 0,
    }
    for n in G_cross.nodes()
]).sort_values("cross_betweenness", ascending=False)

# Scatter: cross-party degree vs. total degree, sized by betweenness
fig_bridge = px.scatter(
    bridge_df.head(100), x="total_degree", y="cross_weighted_degree",
    color="party",
    color_discrete_map={"D": "#2166ac", "R": "#b2182b", "I": "#4daf4a"},
    size="cross_betweenness",
    hover_data=["name", "state", "cross_ratio"],
    labels={"total_degree": "Total Weighted Degree (all cosponsorship)",
            "cross_weighted_degree": "Cross-Party Weighted Degree",
            "cross_betweenness": "Cross-Party Betweenness"},
    title="Bridge Legislators: Who Connects Across Party Lines?",
)
fig_bridge.update_layout(template="plotly_white", height=500)
fig_bridge.show()

print("Top 15 Cross-Party Bridge Legislators:")
bridge_df.head(15)[["name", "party", "state", "cross_betweenness", "cross_ratio"]]

## 2.4 Coalition Stability Across Policy Areas

Do the same sponsors cluster together regardless of topic, or do coalitions reshape by policy area? For each major policy area, run community detection on the area-specific subgraph and measure Jaccard overlap with the full-graph communities.

In [ ]:
# Build policy-area-specific cosponsor graphs and detect communities
cosp_policy = cosp_df.merge(docs_df[["AGORA ID", "Policy_Area"]], on="AGORA ID", how="left")
top_areas = cosp_policy["Policy_Area"].value_counts().head(8).index.tolist()

# Full-graph communities as baseline
full_comms = list(greedy_modularity_communities(G_cc, weight="weight"))
full_comm_sets = [set(c) for c in full_comms]

# Per-area community detection + Jaccard overlap with full communities
stability_results = []
for area in top_areas:
    area_bios = set(cosp_policy[cosp_policy["Policy_Area"] == area]["Cosponsor_BioguideId"])
    # Subgraph for this policy area
    area_nodes = [n for n in G_cc.nodes() if n in area_bios]
    if len(area_nodes) < 10:
        continue
    G_area = G_cc.subgraph(area_nodes).copy()
    
    # Community detection on subgraph
    area_comms = list(greedy_modularity_communities(G_area, weight="weight"))
    
    # Measure best Jaccard overlap between area communities and full communities
    max_jaccards = []
    for ac in area_comms:
        best_j = max(
            len(ac & fc) / len(ac | fc) if len(ac | fc) > 0 else 0
            for fc in full_comm_sets
        )
        max_jaccards.append(best_j)
    
    avg_stability = np.mean(max_jaccards) if max_jaccards else 0
    stability_results.append({
        "Policy Area": area,
        "Subgraph Nodes": len(area_nodes),
        "Subgraph Communities": len(area_comms),
        "Avg Jaccard Overlap": round(avg_stability, 3),
    })

stab_df = pd.DataFrame(stability_results).sort_values("Avg Jaccard Overlap", ascending=True)

fig_stab = px.bar(
    stab_df, x="Avg Jaccard Overlap", y="Policy Area",
    orientation="h",
    color="Avg Jaccard Overlap",
    color_continuous_scale="RdYlGn",
    hover_data=["Subgraph Nodes", "Subgraph Communities"],
    title="Coalition Stability: Do Communities Persist Within Policy Areas?",
    labels={"Avg Jaccard Overlap": "Avg Best Jaccard Overlap with Full-Graph Communities"},
)
fig_stab.update_layout(template="plotly_white", height=400)
fig_stab.show()

print("Higher overlap = coalitions persist across topics")
print("Lower overlap = coalitions reshape by policy area")
stab_df

## 2.5 Community Network Visualization

Force-directed layout of the cosponsor-cosponsor graph, colored by party, with community boundaries visible.

In [ ]:
# Use largest connected component for layout clarity
largest_cc = max(nx.connected_components(G_cc), key=len)
G_viz = G_cc.subgraph(largest_cc).copy()

# Assign community labels
comm_map = {}
for i, comm in enumerate(full_comms):
    for node in comm:
        comm_map[node] = i

pos = nx.spring_layout(G_viz, weight="weight", seed=42, k=0.3, iterations=50)

# Build plotly figure
party_colors = {"D": "#2166ac", "R": "#b2182b", "I": "#4daf4a", "": "#999999"}

# Edges (sample for performance — draw top 20% by weight)
edges_sorted = sorted(G_viz.edges(data=True), key=lambda e: e[2].get("weight", 0), reverse=True)
edge_limit = max(len(edges_sorted) // 5, 500)

fig_comm = go.Figure()
for u, v, d in edges_sorted[:edge_limit]:
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    fig_comm.add_trace(go.Scatter(
        x=[x0, x1, None], y=[y0, y1, None],
        mode="lines", line=dict(width=0.3, color="rgba(180,180,180,0.2)"),
        hoverinfo="skip", showlegend=False,
    ))

# Nodes by party
for party, color in party_colors.items():
    nodes = [n for n in G_viz.nodes() if G_viz.nodes[n].get("party", "") == party]
    if not nodes:
        continue
    fig_comm.add_trace(go.Scatter(
        x=[pos[n][0] for n in nodes],
        y=[pos[n][1] for n in nodes],
        mode="markers",
        marker=dict(size=5, color=color, line=dict(width=0.3, color="white")),
        text=[f"{G_viz.nodes[n].get('name', n)[:30]}<br>Community: {comm_map.get(n, '?')}" for n in nodes],
        hoverinfo="text",
        name=f"Party {party}" if party else "Unknown",
    ))

fig_comm.update_layout(
    title=f"Cosponsor Network (largest component: {len(G_viz)} nodes, top {edge_limit} edges)",
    template="plotly_white", height=600, width=800,
    xaxis=dict(visible=False), yaxis=dict(visible=False),
    legend=dict(x=0.01, y=0.99),
)
fig_comm.show()

## 2.6 Exports

In [ ]:
au.save_df(bi_df, "bipartisanship_by_document")
au.save_df(bridge_df, "bridge_legislators")
au.save_df(stab_df, "coalition_stability")
au.save_df(profiles_df, "community_profiles")

fig_bi_hist.write_html(str(au.OUTPUTS_DIR / "fig_bipartisanship_dist.html"))
fig_bi_box.write_html(str(au.OUTPUTS_DIR / "fig_bipartisanship_by_area.html"))
fig_bridge.write_html(str(au.OUTPUTS_DIR / "fig_bridge_legislators.html"))
fig_stab.write_html(str(au.OUTPUTS_DIR / "fig_coalition_stability.html"))
fig_comm.write_html(str(au.OUTPUTS_DIR / "fig_cosponsor_network.html"))

print("Module 2 exports saved to notebooks/outputs/")